# ACV-LQN - Object Detection with YOLO, DETR, Faster R-CNN - TH03

## Phát hiện đối tượng sử dụng YOLO, DETR, Faster R-CNN

**Nhóm:** Myopia

| MSHV | Họ tên |
|---|---|
| 25C11067 | Nguyễn Thái Thông |
| 25C11035 | Trần Hạ Khánh Duy |



## Mục lục

- [0. Clone repo và checkout nhánh](#0-clone-repo-và-checkout-nhánh)
- [1. Cài đặt thư viện](#1-cài-đặt-thư-viện)
- [2. Tải dataset từ Google Drive](#2-tải-dataset-từ-google-drive)
- [3. Kiểm tra môi trường](#3-kiểm-tra-môi-trường)
- [4. Chuẩn bị dữ liệu](#4-chuẩn-bị-dữ-liệu)
- [5. Sinh hình minh chứng](#5-sinh-hình-minh-chứng)
- [6. Fine-tune YOLO](#6-fine-tune-yolo)
- [7. Fine-tune DETR](#7-fine-tune-detr)
- [8. Fine-tune Faster R-CNN](#8-fine-tune-faster-r-cnn)
- [9. Kết quả](#9-kết-quả)
- [10. Kết luận](#10-kết-luận)


## 0. Clone repo và checkout nhánh

Mục này dùng để khai báo mã nguồn chứa code để thực nghiệm.


In [ ]:
from pathlib import Path

# URL repository
REPO_URL = 'https://github.com/tthongbos/AdvancedMachineLearning.git'
# Nhánh chứa mã nguồn cần sử dụng.
BRANCH = 'main'
# Thư mục clone repository trên Kaggle.
REPO_DIR = Path('/kaggle/working/th03_repository')
# Đường dẫn tương đối tới thư mục project TH03 bên trong repository.
PROJECT_SUBDIR = Path('acv/TH/TH_3/Myopia_25C11067_25C11035_th03')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout {BRANCH}
!git status --short --branch

PROJECT_ROOT = REPO_DIR / PROJECT_SUBDIR
SOURCE_DIR = PROJECT_ROOT / 'source'
SCRIPT_PATH = SOURCE_DIR / 'object_detection.py'
FIGURES_DIR = PROJECT_ROOT.parent / 'doc' / 'figures'

print('Project root:', PROJECT_ROOT)
print('Script path:', SCRIPT_PATH)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'Cannot find script: {SCRIPT_PATH}. Check PROJECT_SUBDIR.')

%cd {PROJECT_ROOT}


## 1. Cài đặt thư viện


In [ ]:
%pip install -q -r {SOURCE_DIR / 'requirements.txt'}


## 2. Tải dataset từ Google Drive

Dataset được lưu dưới dạng file `.zip` trên Google Drive.


In [ ]:
import zipfile

import gdown

DATASET_FILE_ID = '1sww39WoPZ6ICktbJwbSsHTxOQyE7R5hH'
DATA_DIR = Path('/kaggle/working/data')
ARCHIVE_PATH = DATA_DIR / '416x416_aug_chess_pieces.zip'
DATASET_DIR = DATA_DIR / '416x416_aug_chess_pieces'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    gdown.download(id=DATASET_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)

if not DATASET_DIR.exists() and ARCHIVE_PATH.suffix == '.zip':
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)

dataset = DATASET_DIR.resolve()
print('Dataset:', dataset)
print('Dataset exists:', dataset.exists())


## 3. Kiểm tra môi trường


In [ ]:
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset root:', dataset, dataset.exists())


## 4. Chuẩn bị dữ liệu


In [ ]:
!python {SCRIPT_PATH} prepare --dataset {dataset}


## 5. Sinh hình minh chứng


In [ ]:
!python {SCRIPT_PATH} figures --dataset {dataset}


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(FIGURES_DIR / 'dataset_samples_grid.png')))
display(Image(filename=str(FIGURES_DIR / 'split_distribution.png')))
display(Image(filename=str(FIGURES_DIR / 'class_distribution.png')))


## 6. Fine-tune YOLO


In [ ]:
!python {SCRIPT_PATH} yolo --dataset {dataset} --epochs 30 --batch-size 8 --imgsz 416


## 7. Fine-tune DETR


In [ ]:
!python {SCRIPT_PATH} detr --dataset {dataset} --epochs 20 --batch-size 2


## 8. Fine-tune Faster R-CNN


In [ ]:
!python {SCRIPT_PATH} fasterrcnn --dataset {dataset} --epochs 20 --batch-size 2


## 9. Kết quả

Phần này tổng hợp ảnh ground-truth, biểu đồ train/validation, ảnh prediction và ảnh so sánh đồng bộ sau khi fine-tune YOLO, DETR và Faster R-CNN.



In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

results_dir = FIGURES_DIR
report_figures = [
    ('dataset_samples_grid.png', '4 dataset samples'),
    ('sample_ground_truth.png', 'Ground-truth mẫu'),
    ('split_distribution.png', 'Phân bố split'),
    ('class_distribution.png', 'Phân bố lớp'),
    ('yolo_training_curve.png', 'YOLO training curve'),
    ('yolo_confusion_matrix.png', 'YOLO confusion matrix'),
    ('yolo_prediction_1.png', 'YOLO prediction 1'),
    ('yolo_prediction_2.png', 'YOLO prediction 2'),
    ('yolo_prediction_3.png', 'YOLO prediction 3'),
    ('yolo_prediction_4.png', 'YOLO prediction 4'),
    ('yolo_prediction_5.png', 'YOLO prediction 5'),
    ('detr_eval_curve.png', 'DETR evaluation curve'),
    ('detr_confusion_matrix.png', 'DETR confusion matrix'),
    ('detr_prediction_1.png', 'DETR prediction 1'),
    ('detr_prediction_2.png', 'DETR prediction 2'),
    ('detr_prediction_3.png', 'DETR prediction 3'),
    ('detr_prediction_4.png', 'DETR prediction 4'),
    ('detr_prediction_5.png', 'DETR prediction 5'),
    ('faster_rcnn_eval_curve.png', 'Faster R-CNN evaluation curve'),
    ('faster_rcnn_confusion_matrix.png', 'Faster R-CNN confusion matrix'),
    ('faster_rcnn_prediction_1.png', 'Faster R-CNN prediction 1'),
    ('faster_rcnn_prediction_2.png', 'Faster R-CNN prediction 2'),
    ('faster_rcnn_prediction_3.png', 'Faster R-CNN prediction 3'),
    ('faster_rcnn_prediction_4.png', 'Faster R-CNN prediction 4'),
    ('faster_rcnn_prediction_5.png', 'Faster R-CNN prediction 5'),
    ('comparison_case_1.png', 'Comparison case 1'),
    ('comparison_case_2.png', 'Comparison case 2'),
    ('comparison_case_3.png', 'Comparison case 3'),
    ('comparison_case_4.png', 'Comparison case 4'),
    ('comparison_case_5.png', 'Comparison case 5'),
    ('failure_case_1.png', 'Failure case 1'),
    ('failure_case_2.png', 'Failure case 2'),
]

for filename, title in report_figures:
    image_path = results_dir / filename
    if image_path.exists():
        display(Markdown(f'**{title}**'))
        display(Image(filename=str(image_path)))


## 10. Kết luận

Bài thực hành đã xây dựng pipeline phát hiện đối tượng trên bộ dữ liệu Chess Pieces với ba hướng tiếp cận gồm YOLO, DETR và Faster R-CNN. Dữ liệu đã được chuẩn hóa về các định dạng phù hợp cho từng mô hình, đồng thời các hình minh chứng và ảnh dự đoán sau huấn luyện cũng đã được xuất ra để phục vụ phân tích kết quả.

Trong ba mô hình được khảo sát, YOLO có ưu điểm về tốc độ huấn luyện và suy luận, phù hợp cho các thực nghiệm cần triển khai nhanh. DETR đại diện cho hướng tiếp cận dựa trên transformer, có pipeline dự đoán hiện đại và ít phụ thuộc vào các bước hậu xử lý thủ công hơn. Faster R-CNN là mô hình hai giai đoạn có tính ổn định cao, thường cho kết quả đáng tin cậy trên các bài toán phát hiện đối tượng với số lượng lớp không quá lớn.
